In [3]:
import sys

sys.path.append("../src")
import pandas as pd
import geopandas as gpd
import glamos_processing as glamos
import meteoswiss_processing as meteo
import rioxarray
import xarray as xr

In [4]:
gl = glamos.get_data(1982, 2025)
gl

Extracting glacier mass balance...
Extracting glacier geometry...


,id,observation_start,observation_end,mass_balance_annual,coordx,coordy,geometry
0,A10g-05,1982-10-01,1983-09-30,-625,2801722,1192164,"POLYGON ((2802438.84 1192171.652, 2802402.284 ..."
1,A10g-05,1983-10-01,1984-09-30,848,2801722,1192164,"POLYGON ((2802438.84 1192171.652, 2802402.284 ..."
2,A10g-05,1984-10-01,1985-09-30,356,2801722,1192164,"POLYGON ((2802438.84 1192171.652, 2802402.284 ..."
3,A10g-05,1985-10-01,1986-09-30,-166,2801722,1192164,"POLYGON ((2802438.84 1192171.652, 2802402.284 ..."
4,A10g-05,1986-10-01,1987-09-30,-838,2801722,1192164,"POLYGON ((2802438.84 1192171.652, 2802402.284 ..."
...,...,...,...,...,...,...,...
706,E23-18,2020-10-01,2021-09-30,-882,2783300,1143402,"POLYGON ((2783315.901 1143580.76, 2783336.387 ..."
707,E23-18,2021-10-01,2022-09-30,-3617,2783300,1143402,"POLYGON ((2783315.901 1143580.76, 2783336.387 ..."
708,E23-18,2022-10-01,2023-09-30,-1594,2783300,1143402,"POLYGON ((2783315.901 1143580.76, 2783336.387 ..."
709,E23-18,2023-10-01,2024-09-30,-486,2783300,1143402,"POLYGON ((2783315.901 1143580.76, 2783336.387 ..."


In [5]:
gl["id"].unique()

<ArrowStringArray>
['A10g-05', 'A10g-18', 'A14g-16', 'A14p-01', 'A14p-03', 'A50d-01', 'A50i-06',
 'A50i-07', 'A50i-19', 'A51e-08', 'A51e-12', 'A51e-37', 'A54g-03', 'A55f-03',
  'B16-01',  'B22-01',  'B36-26',  'B43-03',  'B45-04',  'B52-20',  'B52-24',
  'B52-29',  'B52-32',  'B52-33',  'B53-14',  'B55-15',  'B56-03',  'B56-14',
  'B73-13',  'B75-12',  'B82-14',  'B82-27',  'B83-03',  'B85-23',  'B93-06',
  'C14-10',  'E23-16',  'E23-18']
Length: 38, dtype: str

In [6]:
poly = gpd.read_parquet(
    "../data/glacier_geometry_2013-2018.parquet",
)
poly

,sgi-id,geometry
0,A10g-04,"POLYGON ((2802224.865 1193097.586, 2802227.799..."
1,A54e-12,"MULTIPOLYGON (((2676375.305 1174502.297, 26763..."
2,A54e-19,"POLYGON ((2673410.261 1172337.403, 2673426.795..."
3,A10g-09,"POLYGON ((2798559.905 1190687.342, 2798585.701..."
4,A54j-01,"POLYGON ((2656707.845 1169768.145, 2656717.484..."
...,...,...
1395,C83-15,"POLYGON ((2772945.74 1133624.726, 2772947.49 1..."
1396,A54e-15,"MULTIPOLYGON (((2672856.354 1174156.594, 26728..."
1397,A54e-14,"POLYGON ((2672857.466 1173500.634, 2672854.581..."
1398,A54e-13,"MULTIPOLYGON (((2673349.861 1172347.237, 26733..."


In [7]:
poly = poly[poly["sgi-id"].isin(gl["id"].unique())]
poly

,sgi-id,geometry
46,A10g-18,"POLYGON ((2793461.105 1182736.64, 2793468.335 ..."
91,B45-04,"POLYGON ((2667270.446 1143066.141, 2667271.278..."
117,A51e-08,"POLYGON ((2690063.92 1161155.872, 2690023.577 ..."
126,A51e-12,"POLYGON ((2688829.981 1161272.992, 2688847.201..."
202,B43-03,"POLYGON ((2674394.494 1167337.068, 2674398.325..."
312,A55f-03,"POLYGON ((2603400.127 1136822.648, 2603436.359..."
360,A14p-03,"MULTIPOLYGON (((2737967.249 1197461.012, 27379..."
362,A14p-01,"MULTIPOLYGON (((2732086.049 1194311.167, 27320..."
391,A14g-16,"POLYGON ((2713117.266 1164204.071, 2713103.776..."
405,B56-14,"POLYGON ((2634812.235 1096335.938, 2634793.649..."


In [8]:
gl["id"].unique()[~gl["id"].unique().isin(poly["sgi-id"])]

<ArrowStringArray>
[]
Length: 0, dtype: str

In [9]:
df = pd.read_csv(
    "https://doi.glamos.ch/data/glacier_list/glacier_list.csv",
    skiprows=9,
    parse_dates=True,
    names=[
        "name",
        "id",
        "coordx",
        "coordy",
        "area",
        "survey_year",
        "length_change_data_available",
        "mass_balance_data_available",
        "volume_change_data_available",
    ],
)

In [10]:
df[df.id.isin(gl["id"].unique()[~gl["id"].unique().isin(poly["sgi-id"])])]

,name,id,coordx,coordy,area,survey_year,length_change_data_available,mass_balance_data_available,volume_change_data_available


In [11]:
gl[gl.id.isin(gl["id"].unique()[~gl["id"].unique().isin(poly["sgi-id"])])]

,id,observation_start,observation_end,mass_balance_annual,coordx,coordy,geometry


Add "jupyter.kernels.trusted": [
        "/path/to/notebook"
    ], to VSCode settings.json and restart VSCode to make this work

In [12]:
poly.explore()

In [13]:
poly.boundary.explore()

In [14]:
poly.bounds

,minx,miny,maxx,maxy
46,2.792970e+06,1.182347e+06,2.793486e+06,1.182902e+06
91,2.666697e+06,1.141713e+06,2.670617e+06,1.145092e+06
117,2.689665e+06,1.161130e+06,2.690083e+06,1.161315e+06
126,2.688743e+06,1.161112e+06,2.689534e+06,1.161776e+06
202,2.671649e+06,1.159189e+06,2.675282e+06,1.167414e+06
312,2.603206e+06,1.135806e+06,2.608323e+06,1.138181e+06
360,2.737346e+06,1.196271e+06,2.738590e+06,1.197484e+06
362,2.730831e+06,1.192777e+06,2.732520e+06,1.194314e+06
391,2.712168e+06,1.164146e+06,2.713515e+06,1.165609e+06
405,2.632209e+06,1.095095e+06,2.635872e+06,1.096590e+06


In [18]:
met = meteo.get_data(1982, 2025, True)
met

OSError: no files to open

In [17]:
met.sel(
    E=slice(2.792728e06, 2.793728e06),
    N=slice(1.182347e06, 1.182902e06),
    time="2020-01-01",
)["TabsM"].values

NameError: name 'met' is not defined

In [ ]:
poly

In [ ]:
poly.loc[46, "geometry"]

In [ ]:
met

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="angle from rectified to skew grid parameter lost in conversion to CF")

In [ ]:
met = met.rio.write_crs("EPSG:2056")

In [ ]:
poly.geometry.iloc[1]

In [ ]:
met.rio.clip([poly.geometry.iloc[1]])

In [ ]:
met.sel(E=gl.coordx.iloc[0], N=gl.coordy.iloc[0], method='nearest')

In [ ]:
met.sel(time=slice(gl.observation_start.iloc[0], gl.observation_end.iloc[0])).mean(
    dim=["N", "E"]
).isel(time=3)

In [ ]:
def get_climate_features(row, climate: xr.Dataset):
    climate = climate.sel(time=slice(row['observation_start'], row['observation_end'])) # Select only relevant timeframe

    try:
        climate = climate.rio.clip([row['geometry']]) # Clip to glacier geometry
        climate = climate.mean(dim=['N', 'E']) # Calculate mean across grids
    except Exception as e:
        print(f'Failed to find climate grid in geometry for glacier id: {row['id']}, obs: {row['observation_start']}')
        print(e)
        print('Falling back to nearest coordinate')

        climate = climate.sel(E=row['coordx'], N=row['coordy'], method='nearest')

    q1 = climate.isel(time=0)
    q2 = climate.isel(time=1)
    q3 = climate.isel(time=2)
    q4 = climate.isel(time=3)

    return pd.Series({
        'q1h_temp': q1['TabsM'].values.item(), 
        'q2h_temp': q2['TabsM'].values.item(), 
        'q3h_temp': q3['TabsM'].values.item(), 
        'q4h_temp': q4['TabsM'].values.item(), 
        'q1h_prec': q1['RhiresM'].values.item(), 
        'q2h_prec': q2['RhiresM'].values.item(), 
        'q3h_prec': q3['RhiresM'].values.item(), 
        'q4h_prec': q4['RhiresM'].values.item()})
    

In [ ]:
gl.apply(get_climate_features, axis=1, args=(met,))

In [ ]:
gl = glamos.get_data(1982, 2025)
gl

In [ ]:
climate = meteo.get_data(1982, 2025)
climate

In [ ]:
gl_climate = meteo.get_climate_features(gl, climate)
gl_climate

In [ ]:
gl_climate.describe()

In [ ]:
gl_climate.isna().any()